# 12. Supervised encoder baseline (DeBERTa-v3-base)

Four classification heads (one per rubric criterion) on a shared DeBERTa-v3-base encoder, trained on the
training questions; learning rate chosen on the validation questions (seed 42), then two more seeds with
the chosen rate; the test set is predicted once for all three seeds. Runtime: Google Colab GPU (A100).

0. Upload `src/metrics.py` and `data/Data.xlsx` to the Drive folder `MyDrive/Finetuning/`.
1. Run cells 1-4 (setup, data, model, training code).
2. Cell 5 = smoke test (tiny, validation only).
3. Cell 6 = learning-rate search on validation (seed 42).
4. Cell 7 = two more seeds with the chosen learning rate.
5. Cell 8 = the single test-set prediction (all three seeds).

Saved outputs are in `results/deberta/`.


In [ ]:
# 1. Setup
!pip install -q openpyxl sentencepiece protobuf
import os, re, json, time, random, platform, numpy as np, pandas as pd, torch, torch.nn as nn
import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from google.colab import drive
drive.mount('/content/drive')
import sys; sys.path.insert(0, '/content/drive/MyDrive/Finetuning')   # upload src/metrics.py here
import metrics as M                                                    # shared metrics (src/metrics.py)
assert M.qwk([0, 10], [10, 0]) == -1.0 and M.qwk([0, 5, 10], [0, 5, 10]) == 1.0   # known answers

DATA = '/content/drive/MyDrive/Finetuning/Data.xlsx'          # same file as the fine-tuning notebook
OUT = '/content/drive/MyDrive/Finetuning/bert_output'
CKPT = '/content/bert_ckpt'                                   # local disk, not Drive
os.makedirs(OUT, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
MODEL = 'microsoft/deberta-v3-base'
MAX_LEN, BATCH, MAX_EPOCHS, PATIENCE = 512, 16, 10, 2
LRS, SEEDS = [1e-5, 2e-5, 3e-5], [42, 43, 44]
S = ['Clarity_Score', 'Terminology_Score', 'Coverage_Score', 'Accuracy_Score']
N_CLASSES = [3, 3, 3, 5]
dev = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
# 2. Data: same split as the paper (notebooks 04 and 08), same text preparation as February (notebook 08)
TEST = [4, 8, 18, 19, 21, 22, 25, 31, 45, 55, 57, 68, 69, 75, 80, 81, 84]
VAL = [16, 26, 32, 34, 37, 39, 42, 48, 50, 51, 54, 60, 63, 66, 71, 72, 78]

def clean_text(text):
    """Clean HTML and bullet formatting from text."""
    if not isinstance(text, str):
        return '' if pd.isna(text) else str(text)
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'[\u2022\u00A5]\s*\d*\s*', '- ', text)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return '\n'.join(lines)

df = pd.read_excel(DATA)
df['Q_num'] = df['ID'].apply(lambda x: int(str(x).split(' ')[0].replace('Q', '')))
df['seg1'] = df['Question'].astype(str).str.strip() + '\n' + df['Model Answer'].apply(clean_text)   # question + model answer
df['seg2'] = df['Answer Text'].astype(str).str.strip()                                               # student answer
split = {'train': df[~df.Q_num.isin(TEST + VAL)], 'validation': df[df.Q_num.isin(VAL)], 'test': df[df.Q_num.isin(TEST)]}
for k, d in split.items(): print(k, len(d), 'answers,', d.Q_num.nunique(), 'questions')
assert [len(d) for d in split.values()] == [1530, 510, 510]

In [ ]:
# 3. Tokenising (student answer is cut last) and the model (shared encoder + four heads)
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(d):
    items, cut, checked = [], 0, 0
    budget = MAX_LEN - 3                                # [CLS] A [SEP] B [SEP]
    for a, b, y in zip(d.seg1, d.seg2, d[S].values.astype(int)):
        ia, ib = tok(a, add_special_tokens=False)['input_ids'], tok(b, add_special_tokens=False)['input_ids']
        if len(ia) + len(ib) > budget:
            cut += 1
            ia = ia[:max(budget - len(ib), 0)]          # cut question + model answer first
            ib = ib[:budget - len(ia)]                  # student answer cut only if still too long
        ids = [tok.cls_token_id] + ia + [tok.sep_token_id] + ib + [tok.sep_token_id]
        tt = [0] * (len(ia) + 2) + [1] * (len(ib) + 1)
        if checked < 50 and len(ids) < MAX_LEN:         # must equal the tokenizer's own pair encoding
            ref = tok(a, b)
            assert ids == ref['input_ids'], 'input format differs from tokenizer'
            if 'token_type_ids' in ref: assert tt == ref['token_type_ids'], 'token types differ'
            checked += 1
        items.append((ids, tt, y))
    return items, cut

ENC = {}
for k, d in split.items():
    ENC[k], cut = encode(d)
    print(f'{k}: {cut} of {len(d)} inputs cut to {MAX_LEN} tokens; longest {max(len(i[0]) for i in ENC[k])} tokens')

def batches(items, shuffle, gen=None):
    order = torch.randperm(len(items), generator=gen).tolist() if shuffle else list(range(len(items)))
    for i in range(0, len(order), BATCH):
        b = [items[j] for j in order[i:i + BATCH]]
        L = max(len(x[0]) for x in b)
        ids = torch.tensor([x[0] + [tok.pad_token_id] * (L - len(x[0])) for x in b])
        mask = torch.tensor([[1] * len(x[0]) + [0] * (L - len(x[0])) for x in b])
        tt = torch.tensor([(x[1] or [0] * len(x[0])) + [0] * (L - len(x[0])) for x in b])
        y = torch.tensor(np.array([x[2] for x in b]))
        yield ids.to(dev), mask.to(dev), tt.to(dev), y.to(dev)

class Scorer(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL, dtype=torch.float32)   # fp32 (rule §7)
        h = self.enc.config.hidden_size
        self.drop = nn.Dropout(self.enc.config.hidden_dropout_prob)
        self.heads = nn.ModuleList([nn.Linear(h, k) for k in N_CLASSES])
    def forward(self, ids, mask, tt):
        cls = self.enc(input_ids=ids, attention_mask=mask, token_type_ids=tt).last_hidden_state[:, 0]
        cls = self.drop(cls)
        return [head(cls) for head in self.heads]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# 4. Training and prediction
def predict(model, items):
    model.eval(); out = []
    with torch.no_grad():
        for ids, mask, tt, _ in batches(items, shuffle=False):
            out.append(torch.stack([l.argmax(-1) for l in model(ids, mask, tt)], 1).cpu())
    return torch.cat(out).numpy()                                   # (n, 4) predicted class = score

def composite_qwk(items, pred):
    return M.qwk(np.array([x[2] for x in items]).sum(1), pred.sum(1))

def train(lr, seed, items_train, items_val, tag, max_epochs=MAX_EPOCHS):
    set_seed(seed)
    model = Scorer().to(dev)
    model.enc.gradient_checkpointing_enable()           # saves GPU memory; same results
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = max_epochs * ((len(items_train) + BATCH - 1) // BATCH)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    ce, gen = nn.CrossEntropyLoss(), torch.Generator().manual_seed(seed)
    best, best_ep, wait, hist, t0 = -2, 0, 0, [], time.time()
    for ep in range(1, max_epochs + 1):
        model.train(); tot = 0
        for ids, mask, tt, y in batches(items_train, shuffle=True, gen=gen):
            loss = sum(ce(l, y[:, k]) for k, l in enumerate(model(ids, mask, tt)))   # four heads, equal weights
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); tot += loss.item() * len(y)
        q = composite_qwk(items_val, predict(model, items_val))
        hist.append({'epoch': ep, 'train_loss': tot / len(items_train), 'val_qwk': q})
        print(f'  {tag} epoch {ep}: train loss {tot / len(items_train):.4f}  val composite QWK {q:.4f}')
        if q > best:                                                # ties keep the earlier epoch
            best, best_ep, wait = q, ep, 0
            torch.save(model.state_dict(), f'{CKPT}/{tag}.pt')
        else:
            wait += 1
            if wait >= PATIENCE: break
    run = {'tag': tag, 'lr': lr, 'seed': seed, 'best_epoch': best_ep, 'best_val_qwk': best,
           'minutes': round((time.time() - t0) / 60, 1), 'history': hist}
    json.dump(run, open(f'{OUT}/{tag}.json', 'w'), indent=2)
    del model; torch.cuda.empty_cache()
    return run

In [ ]:
# 5. SMOKE TEST: 2 epochs on the 64 LONGEST training answers, validation only (no test data). Checks code and memory.
longest = sorted(ENC['train'], key=lambda x: -len(x[0]))[:64]
smoke = train(2e-5, 42, longest, ENC['validation'][:64], 'smoke', max_epochs=2)
print('peak GPU memory %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))
print('smoke test OK:', smoke['best_val_qwk'])

In [ ]:
# 6. Learning-rate search on VALIDATION only, seed 42. Ties go to the smaller learning rate.
search = [train(lr, 42, ENC['train'], ENC['validation'], f'lr{lr:g}_seed42') for lr in LRS]
best_q = max(r['best_val_qwk'] for r in search)
LR = min(r['lr'] for r in search if r['best_val_qwk'] == best_q)
print({r['lr']: round(r['best_val_qwk'], 4) for r in search}, '-> chosen LR', LR)
json.dump({'chosen_lr': LR, 'search': [{k: r[k] for k in ['lr', 'best_epoch', 'best_val_qwk']} for r in search]},
          open(f'{OUT}/lr_choice.json', 'w'), indent=2)

In [ ]:
# 7. Two more seeds with the chosen learning rate (the seed-42 search run counts as the first)
runs = {42: next(r for r in search if r['lr'] == LR)}
for s in [43, 44]:
    runs[s] = train(LR, s, ENC['train'], ENC['validation'], f'lr{LR:g}_seed{s}')
med = sorted(runs[s]['best_val_qwk'] for s in SEEDS)[1]
MEDIAN_SEED = min(s for s in SEEDS if runs[s]['best_val_qwk'] == med)   # ties go to the smaller seed
print({s: round(runs[s]['best_val_qwk'], 4) for s in SEEDS}, '-> median-validation seed', MEDIAN_SEED)

In [ ]:
# 8. THE ONE TEST-SET PREDICTION. Run once, after cells 6 and 7.
assert not os.path.exists(f'{OUT}/test_done.json'), 'test already predicted once'
d = split['test'].reset_index(drop=True); TQ = {}
for s in SEEDS:
    m = Scorer().to(dev); m.load_state_dict(torch.load(f'{CKPT}/lr{LR:g}_seed{s}.pt')); 
    p = predict(m, ENC['test']); del m; torch.cuda.empty_cache()
    res = pd.DataFrame({'student_id': d.ID, 'Human_OffTopic': d.off_topic})
    for k, c in enumerate(S):
        n = c.replace('_Score', '')
        res[f'Human_{n}'], res[f'LLM_{n}'] = d[c].astype(int).values, p[:, k]
    res['Human_Final'] = d[S].sum(1).values / 2; res['LLM_Final'] = p.sum(1) / 2
    res.to_csv(f'{OUT}/test_results_seed{s}.csv', index=False)
    TQ[s] = M.qwk(d[S].sum(1).astype(int).values, p.sum(1))
    print(f'seed {s}: test composite QWK {TQ[s]:.4f}')
print(f'mean {np.mean(list(TQ.values())):.4f}, range {min(TQ.values()):.4f} to {max(TQ.values()):.4f}')
json.dump({'chosen_lr': LR, 'median_seed_for_family_test': MEDIAN_SEED,
           'val_qwk_by_seed': {s: runs[s]['best_val_qwk'] for s in SEEDS}, 'test_qwk_by_seed': TQ,
           'best_epoch_by_seed': {s: runs[s]['best_epoch'] for s in SEEDS},
           'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'transformers': transformers.__version__,
           'python': platform.python_version(), 'date': time.strftime('%Y-%m-%d %H:%M')},
          open(f'{OUT}/test_done.json', 'w'), indent=2)
print('Saved to', OUT)